# Optimización de Modelos Conjunto Soleado por GMM

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
#Importaciones
import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import MinMaxScaler

#Lectura de datos
datos = pd.read_excel('/content/drive/MyDrive/Tesina/03_Clusterizacion_CTNET.xlsx')
datos.head(24)

,Fecha,Generación,Temperatura,Humedad Relativa,Índice UV,Hora,Cluster KMeans,Cluster GMM
0,2022-09-01 00:00:00,0.000000,19,77,0,0,Noche,Noche
1,2022-09-01 01:00:00,0.000000,19,82,0,1,Noche,Noche
2,2022-09-01 02:00:00,0.000000,18,85,0,2,Noche,Noche
3,2022-09-01 03:00:00,0.000000,18,87,0,3,Noche,Noche
4,2022-09-01 04:00:00,0.000000,18,88,0,4,Noche,Noche
5,2022-09-01 05:00:00,0.000000,17,86,0,5,Noche,Noche
6,2022-09-01 06:00:00,0.000000,18,89,0,6,Soleado,Lluvioso
7,2022-09-01 07:00:00,6.584959,18,95,0,7,Soleado,Lluvioso
8,2022-09-01 08:00:00,560.422022,18,100,0,8,Soleado,Lluvioso
9,2022-09-01 09:00:00,7720.582326,18,100,1,9,Soleado,Lluvioso


In [3]:
datos["Generacion_prev_hour"] = datos["Generación"].shift(1)
datos["Generacion_prev_day"] = datos["Generación"].shift(24)
datos = datos.dropna(how="any", axis= 0)

Definimos X y y

In [4]:
datos_dia = datos[datos["Cluster GMM"] == "Soleado"].copy()
datos_dia.head(10)

,Fecha,Generación,Temperatura,Humedad Relativa,Índice UV,Hora,Cluster KMeans,Cluster GMM,Generacion_prev_hour,Generacion_prev_day
60,2022-09-03 12:00:00,17723.695569,21,58,4,12,Nublado,Soleado,11246.659307,27523.885172
62,2022-09-03 14:00:00,21548.984244,24,42,6,14,Nublado,Soleado,20400.000000,28500.000000
63,2022-09-03 15:00:00,25500.000000,24,39,7,15,Nublado,Soleado,21548.984244,24647.568577
64,2022-09-03 16:00:00,23122.803757,25,41,6,16,Nublado,Soleado,25500.000000,25500.000000
84,2022-09-04 12:00:00,22850.891263,21,66,6,12,Nublado,Soleado,17646.593401,17723.695569
85,2022-09-04 13:00:00,27000.000000,22,58,10,13,Nublado,Soleado,22850.891263,20400.000000
86,2022-09-04 14:00:00,25168.164594,23,51,12,14,Nublado,Soleado,27000.000000,21548.984244
87,2022-09-04 15:00:00,24300.000000,23,47,10,15,Nublado,Soleado,25168.164594,25500.000000
88,2022-09-04 16:00:00,19686.387416,24,46,7,16,Nublado,Soleado,24300.000000,23122.803757
109,2022-09-05 13:00:00,30000.000000,21,60,6,13,Nublado,Soleado,25558.205239,27000.000000


In [5]:
columns = datos_dia.drop(columns=["Fecha", "Generación", "Cluster KMeans", "Cluster GMM"]).columns

In [6]:
X = datos_dia[columns]
X

,Temperatura,Humedad Relativa,Índice UV,Hora,Generacion_prev_hour,Generacion_prev_day
60,21,58,4,12,11246.659307,27523.885172
62,24,42,6,14,20400.000000,28500.000000
63,24,39,7,15,21548.984244,24647.568577
64,25,41,6,16,25500.000000,25500.000000
84,21,66,6,12,17646.593401,17723.695569
...,...,...,...,...,...,...
18254,23,46,9,13,26286.000000,25559.000000
18255,25,39,8,14,25653.000000,25579.000000
18277,21,51,6,12,26323.000000,26286.000000
18278,23,44,7,13,26277.000000,25653.000000


In [7]:
y = datos_dia[['Generación']]
y

,Generación
60,17723.695569
62,21548.984244
63,25500.000000
64,23122.803757
84,22850.891263
...,...
18254,25653.000000
18255,25362.000000
18277,26277.000000
18278,25598.000000


Dividimos entrenamiento, validación y prueba

In [8]:
train_size = int(0.7 * len(X))
val_size = int(0.85 * len(X))

In [9]:
# Entrenamiento, validación y prueba, 75, 15 y 15
X_train, y_train =  X.iloc[:train_size, :], y.iloc[:train_size, :]
X_val, y_val = X.iloc[train_size:val_size, :], y.iloc[train_size:val_size, :]
X_test, y_test = X.iloc[val_size:, :],  y.iloc[val_size:,:]

print(f'X_train: {len(X_train)}, y_train: {len(y_train)}')
print(f'X_val: {len(X_val)}, y_val: {len(y_val)}')
print(f'X_test: {len(X_test)}, y_test: {len(y_test)}')

X_train: 1388, y_train: 1388
X_val: 298, y_val: 298
X_test: 298, y_test: 298


## Escalar con MinMaxScaler

In [10]:
from sklearn.preprocessing import MinMaxScaler

In [11]:
x_scaler = MinMaxScaler().fit(X_train)
x_scaler

MinMaxScaler()

In [12]:
X_train_scaled = x_scaler.transform(X_train)
print(X_train_scaled)
print(X_train_scaled.shape)

[[0.2        0.79104478 0.         0.28571429 0.37488864 0.91746284]
 [0.35       0.55223881 0.2        0.57142857 0.68       0.95      ]
 [0.35       0.50746269 0.3        0.71428571 0.71829947 0.82158562]
 ...
 [0.35       0.79104478 0.1        0.         0.8229     0.91186667]
 [0.5        0.58208955 0.5        0.14285714 0.90826667 0.9442    ]
 [0.65       0.40298507 0.8        0.28571429 0.94696667 0.93683333]]
(1388, 6)


In [13]:
X_train_scaled_df = pd.DataFrame(X_train_scaled, index=X_train.index, columns=X_train.columns)
X_train_scaled_df

,Temperatura,Humedad Relativa,Índice UV,Hora,Generacion_prev_hour,Generacion_prev_day
60,0.20,0.791045,0.0,0.285714,0.374889,0.917463
62,0.35,0.552239,0.2,0.571429,0.680000,0.950000
63,0.35,0.507463,0.3,0.714286,0.718299,0.821586
64,0.40,0.537313,0.2,0.857143,0.850000,0.850000
84,0.20,0.910448,0.2,0.285714,0.588220,0.590790
...,...,...,...,...,...,...
15015,0.70,0.402985,0.8,0.571429,0.928933,0.932600
15016,0.75,0.328358,0.5,0.714286,0.932600,0.965300
15035,0.35,0.791045,0.1,0.000000,0.822900,0.911867
15036,0.50,0.582090,0.5,0.142857,0.908267,0.944200


In [14]:
X_val_scaled = x_scaler.transform(X_val)
print(X_val_scaled)
print(X_val_scaled.shape)

[[0.75       0.29850746 1.         0.42857143 0.9369     0.92893333]
 [0.8        0.25373134 0.8        0.57142857 0.92893333 0.9326    ]
 [0.9        0.20895522 0.5        0.71428571 0.9326     0.97063333]
 ...
 [0.45       0.6119403  0.5        0.71428571 0.95016667 0.8617    ]
 [0.2        0.94029851 0.4        0.14285714 0.89683333 0.92476667]
 [0.3        0.82089552 0.8        0.28571429 0.92476667 0.93596667]]
(298, 6)


In [15]:
X_val_scaled_df = pd.DataFrame(X_val_scaled, index=X_val.index, columns=X_val.columns)
X_val_scaled_df

,Temperatura,Humedad Relativa,Índice UV,Hora,Generacion_prev_hour,Generacion_prev_day
15038,0.75,0.298507,1.0,0.428571,0.936900,0.928933
15039,0.80,0.253731,0.8,0.571429,0.928933,0.932600
15040,0.90,0.208955,0.5,0.714286,0.932600,0.970633
15059,0.40,0.701493,0.1,0.000000,0.693167,0.908267
15060,0.55,0.507463,0.5,0.142857,0.727967,0.946967
...,...,...,...,...,...,...
16526,0.35,0.716418,1.0,0.428571,0.935967,0.848067
16527,0.45,0.641791,0.8,0.571429,0.943800,0.851767
16528,0.45,0.611940,0.5,0.714286,0.950167,0.861700
16548,0.20,0.940299,0.4,0.142857,0.896833,0.924767


In [16]:
X_test_scaled = x_scaler.transform(X_test)
print(X_test_scaled)
print(X_test_scaled.shape)

[[0.35       0.73134328 0.8        0.42857143 0.9375     0.9438    ]
 [0.45       0.67164179 0.8        0.57142857 0.95086667 0.95016667]
 [0.45       0.64179104 0.5        0.71428571 0.95153333 0.95743333]
 ...
 [0.2        0.68656716 0.2        0.28571429 0.87743333 0.8762    ]
 [0.3        0.58208955 0.3        0.42857143 0.8759     0.8551    ]
 [0.35       0.49253731 0.4        0.57142857 0.85326667 0.8454    ]]
(298, 6)


In [17]:
X_test_scaled_df = pd.DataFrame(X_test_scaled, index=X_test.index, columns=X_test.columns)
X_test_scaled_df

,Temperatura,Humedad Relativa,Índice UV,Hora,Generacion_prev_hour,Generacion_prev_day
16550,0.35,0.731343,0.8,0.428571,0.937500,0.943800
16551,0.45,0.671642,0.8,0.571429,0.950867,0.950167
16552,0.45,0.641791,0.5,0.714286,0.951533,0.957433
16572,0.25,0.895522,0.2,0.142857,0.933400,0.924767
16573,0.30,0.791045,0.8,0.285714,0.852400,0.937500
...,...,...,...,...,...,...
18254,0.30,0.611940,0.5,0.428571,0.876200,0.851967
18255,0.40,0.507463,0.4,0.571429,0.855100,0.852633
18277,0.20,0.686567,0.2,0.285714,0.877433,0.876200
18278,0.30,0.582090,0.3,0.428571,0.875900,0.855100


In [18]:
x_scaller_all = MinMaxScaler().fit(X)
print(x_scaller_all)

MinMaxScaler()


In [19]:
X_scaled = x_scaller_all.transform(X)
print(X_scaled)
print(X_scaled.shape)

[[0.19047619 0.76811594 0.         0.28571429 0.37488864 0.91746284]
 [0.33333333 0.53623188 0.2        0.57142857 0.68       0.95      ]
 [0.33333333 0.49275362 0.3        0.71428571 0.71829947 0.82158562]
 ...
 [0.19047619 0.66666667 0.2        0.28571429 0.87743333 0.8762    ]
 [0.28571429 0.56521739 0.3        0.42857143 0.8759     0.8551    ]
 [0.33333333 0.47826087 0.4        0.57142857 0.85326667 0.8454    ]]
(1984, 6)


In [20]:
X_scaled_df = pd.DataFrame(X_scaled, index=X.index, columns=X.columns)
X_scaled_df

,Temperatura,Humedad Relativa,Índice UV,Hora,Generacion_prev_hour,Generacion_prev_day
60,0.190476,0.768116,0.0,0.285714,0.374889,0.917463
62,0.333333,0.536232,0.2,0.571429,0.680000,0.950000
63,0.333333,0.492754,0.3,0.714286,0.718299,0.821586
64,0.380952,0.521739,0.2,0.857143,0.850000,0.850000
84,0.190476,0.884058,0.2,0.285714,0.588220,0.590790
...,...,...,...,...,...,...
18254,0.285714,0.594203,0.5,0.428571,0.876200,0.851967
18255,0.380952,0.492754,0.4,0.571429,0.855100,0.852633
18277,0.190476,0.666667,0.2,0.285714,0.877433,0.876200
18278,0.285714,0.565217,0.3,0.428571,0.875900,0.855100


In [21]:
y_scaler = MinMaxScaler().fit(y_train)
print(y_scaler)

MinMaxScaler()


In [22]:
y_train_scaled = y_scaler.transform(y_train)
print(y_train_scaled)
print(y_train_scaled.shape)

[[0.59078985]
 [0.71829947]
 [0.85      ]
 ...
 [0.90826667]
 [0.94696667]
 [0.9369    ]]
(1388, 1)


In [23]:
y_train_scaled_df = pd.DataFrame(y_train_scaled, index=y_train.index, columns=y_train.columns)
y_train_scaled_df

,Generación
60,0.590790
62,0.718299
63,0.850000
64,0.770760
84,0.761696
...,...
15015,0.932600
15016,0.970633
15035,0.908267
15036,0.946967


In [24]:
y_val_scaled = y_scaler.transform(y_val)
print(y_val_scaled)
print(y_val_scaled.shape)

[[0.92893333]
 [0.9326    ]
 [0.97063333]
 [0.72796667]
 [0.751     ]
 [0.9429    ]
 [0.93506667]
 [0.885     ]
 [0.86473333]
 [0.8983    ]
 [0.95563333]
 [0.98136667]
 [0.97883333]
 [0.94683333]
 [0.92646667]
 [0.9025    ]
 [0.9426    ]
 [0.93683333]
 [0.93753333]
 [0.9363    ]
 [0.92646667]
 [0.90996667]
 [0.9332    ]
 [0.94056667]
 [0.93186667]
 [0.95676667]
 [0.83383333]
 [0.9332    ]
 [0.9537    ]
 [0.85843333]
 [0.77453333]
 [0.76263333]
 [0.9332    ]
 [0.9479    ]
 [0.95776667]
 [0.96316667]
 [0.9624    ]
 [0.9332    ]
 [0.9484    ]
 [0.96203333]
 [0.96876667]
 [0.95776667]
 [0.93553333]
 [0.94413333]
 [0.84376667]
 [0.84483333]
 [0.96416667]
 [0.94396667]
 [0.97616667]
 [0.97016667]
 [0.97313333]
 [0.94583333]
 [0.9501    ]
 [0.93826667]
 [0.9582    ]
 [0.95276667]
 [0.94963333]
 [0.95856667]
 [0.9593    ]
 [0.94006667]
 [0.96873333]
 [0.95253333]
 [0.95486667]
 [0.97816667]
 [0.95946667]
 [0.91103333]
 [0.94243333]
 [0.94176667]
 [0.93916667]
 [0.93993333]
 [0.9288    ]
 [0.72

In [25]:
y_val_scaled_df = pd.DataFrame(y_val_scaled, index=y_val.index, columns=y_val.columns)
y_val_scaled_df

,Generación
15038,0.928933
15039,0.932600
15040,0.970633
15059,0.727967
15060,0.751000
...,...
16526,0.943800
16527,0.950167
16528,0.957433
16548,0.924767


In [26]:
y_test_scaled = y_scaler.transform(y_test)
print(y_test_scaled)
print(y_test_scaled.shape)

[[0.95086667]
 [0.95153333]
 [0.95743333]
 [0.8524    ]
 [0.8321    ]
 [0.9522    ]
 [0.6613    ]
 [0.65823333]
 [0.39263333]
 [0.39453333]
 [0.46656667]
 [0.4486    ]
 [0.51253333]
 [0.66073333]
 [0.66416667]
 [0.688     ]
 [0.66283333]
 [0.92123333]
 [0.9215    ]
 [0.91553333]
 [0.91133333]
 [0.8989    ]
 [0.92446667]
 [0.92853333]
 [0.91553333]
 [0.91063333]
 [0.7407    ]
 [0.737     ]
 [0.83793333]
 [0.824     ]
 [0.82563333]
 [0.92456667]
 [0.94646667]
 [0.95036667]
 [0.94076667]
 [0.92476667]
 [0.92123333]
 [0.91896667]
 [0.91553333]
 [0.81956667]
 [0.92196667]
 [0.92623333]
 [0.92043333]
 [0.9242    ]
 [0.9156    ]
 [0.92236667]
 [0.8345    ]
 [0.72543333]
 [0.82626667]
 [0.9101    ]
 [0.92196667]
 [0.92103333]
 [0.9068    ]
 [0.91263333]
 [0.90873333]
 [0.92196667]
 [0.92593333]
 [0.90816667]
 [0.92106667]
 [0.9109    ]
 [0.92196667]
 [0.92456667]
 [0.90863333]
 [0.9157    ]
 [0.91783333]
 [0.92613333]
 [0.92293333]
 [0.9093    ]
 [0.90856667]
 [0.90463333]
 [0.8344    ]
 [0.92

In [27]:
y_test_scaled_df = pd.DataFrame(y_test_scaled, index=y_test.index, columns=y_test.columns)
y_test_scaled_df

,Generación
16550,0.950867
16551,0.951533
16552,0.957433
16572,0.852400
16573,0.832100
...,...
18254,0.855100
18255,0.845400
18277,0.875900
18278,0.853267


In [28]:
y_scaller_all = MinMaxScaler().fit(y)
print(y_scaller_all)

MinMaxScaler()


In [29]:
y_scaled = y_scaller_all.transform(y)
print(y_scaled)
print(y_scaled.shape)

[[0.59078985]
 [0.71829947]
 [0.85      ]
 ...
 [0.8759    ]
 [0.85326667]
 [0.84663333]]
(1984, 1)


In [30]:
y_scaled_df = pd.DataFrame(y_scaled, index=y.index, columns=y.columns)
y_scaled_df

,Generación
60,0.590790
62,0.718299
63,0.850000
64,0.770760
84,0.761696
...,...
18254,0.855100
18255,0.845400
18277,0.875900
18278,0.853267


## Preparación para Redes Neuronales

In [31]:
import numpy as np
import pandas as pd

def create_sliding_window_with_index(data_X, data_y, lookback):
    X, y, indices = [], [], []

    # Asegurar que `data_y` tiene los mismos índices que `data_X`
    data_y = data_y.reindex(data_X.index)

    max_index = len(data_X) - lookback

    for i in range(max_index):
        X.append(data_X.iloc[i:i + lookback].values)  # Ventana de entrada

        # Obtener el índice correcto en `data_y`
        y_index = data_X.index[i + lookback]

        # Extraer el valor correspondiente de `data_y`
        if y_index in data_y.index:
            y_value = data_y.loc[y_index]
            if isinstance(y_value, pd.Series):  # Si devuelve una serie, extraer el valor
                y_value = y_value.iloc[0]
        else:
            y_value = np.nan  # Si no está, asignamos NaN

        y.append(y_value)
        indices.append(y_index)  # 🔹 Guardamos el índice original de `data_y`

    # Convertimos `X` en un array y `y` en DataFrame conservando sus índices originales
    X_array = np.array(X)
    y_df = pd.DataFrame(y, index=indices, columns=['y'])  # 🔹 Conservamos los índices originales

    return X_array, y_df


In [32]:
lookback = 48  # Puedes ajustar a 24, 72, etc.

# Aplicar la ventana deslizante a cada conjunto
X_train_windowed, y_train_windowed = create_sliding_window_with_index(X_train_scaled_df, y_train_scaled_df, lookback)
X_val_windowed, y_val_windowed = create_sliding_window_with_index(X_val_scaled_df, y_val_scaled_df, lookback)
X_test_windowed, y_test_windowed = create_sliding_window_with_index(X_test_scaled_df, y_test_scaled_df, lookback)


In [33]:
print(f'X_train: {X_train_windowed.shape}, y_train: {y_train_windowed.shape}')
print(f'X_val: {X_val_windowed.shape}, y_val: {y_val_windowed.shape}')
print(f'X_test: {X_test_windowed.shape}, y_test: {y_test_windowed.shape}')

X_train: (1340, 48, 6), y_train: (1340, 1)
X_val: (250, 48, 6), y_val: (250, 1)
X_test: (250, 48, 6), y_test: (250, 1)


## Optuna

In [34]:
pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.6/383.6 kB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.8/231.8 kB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.5/78.5 kB 9.0 MB/s eta 0:00:00


In [35]:
from lightgbm import LGBMRegressor
from sklearn.ensemble import RandomForestRegressor
import optuna
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import cross_val_score
import seaborn as sns
from sklearn.metrics import mean_absolute_percentage_error as mean_absolute_percentage_error
from sklearn.metrics import mean_absolute_error as mean_absolute_error
from sklearn.metrics import mean_squared_error as mean_squared_error
from sklearn.metrics import r2_score as r2_score
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping

### Optimización de LightGBM

In [36]:
import optuna
import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_squared_error

# Definir la función de optimización
def objective(trial):
    # Espacio de búsqueda para los hiperparámetros
    num_leaves = trial.suggest_int('num_leaves', 10, 1000)
    subsample = trial.suggest_float('subsample', 0.1, 1.0)
    colsample_bytree = trial.suggest_float('colsample_bytree', 0.1, 1.0)
    min_data_in_leaf = trial.suggest_int('min_data_in_leaf', 10, 100)

    # Modelo LightGBM con los hiperparámetros sugeridos
    model = LGBMRegressor(
        num_leaves=num_leaves,
        subsample=subsample,
        colsample_bytree=colsample_bytree,
        min_data_in_leaf=min_data_in_leaf,
        random_state=42
    )

    # Entrenamos el modelo con los datos de entrenamiento
    model.fit(X_train_scaled_df, y_train_scaled_df)

    # Predicciones en el conjunto de validación
    y_pred = model.predict(X_val_scaled_df)
    rmse = mean_squared_error(y_val_scaled_df, y_pred)

    return rmse  # Optuna minimizará este valor

# Ejecutar la optimización con 50 iteraciones
LightGBM_study = optuna.create_study(direction="minimize")
LightGBM_study.optimize(objective, n_trials=50)

# Obtener los mejores hiperparámetros
best_params = LightGBM_study.best_params
print("Mejores hiperparámetros:", best_params)

[I 2025-03-14 13:14:21,313] A new study created in memory with name: no-name-0b31b9dc-2663-458f-a8f3-0c9ea77d4492
[I 2025-03-14 13:14:21,441] Trial 0 finished with value: 0.0061205471094448365 and parameters: {'num_leaves': 135, 'subsample': 0.14270058744697572, 'colsample_bytree': 0.8001366758313877, 'min_data_in_leaf': 21}. Best is trial 0 with value: 0.0061205471094448365.
[I 2025-03-14 13:14:21,485] Trial 1 finished with value: 0.004740695291357183 and parameters: {'num_leaves': 140, 'subsample': 0.8196875037233156, 'colsample_bytree': 0.9898448260996575, 'min_data_in_leaf': 49}. Best is trial 1 with value: 0.004740695291357183.


[LightGBM] [Warning] min_data_in_leaf is set=21, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=21
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Warning] min_data_in_leaf is set=21, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=21
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000874 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 614
[LightGBM] [Info] Number of data points in the train set: 1388, number of used features: 6
[LightGBM] [Info] Start training from score 0.884575
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with posi

[I 2025-03-14 13:14:21,558] Trial 2 finished with value: 0.005282192904611565 and parameters: {'num_leaves': 346, 'subsample': 0.7125993981289098, 'colsample_bytree': 0.9753457011131746, 'min_data_in_leaf': 26}. Best is trial 1 with value: 0.004740695291357183.
[I 2025-03-14 13:14:21,595] Trial 3 finished with value: 0.008103743575596397 and parameters: {'num_leaves': 404, 'subsample': 0.2921365011629893, 'colsample_bytree': 0.9337656003902293, 'min_data_in_leaf': 79}. Best is trial 1 with value: 0.004740695291357183.
[I 2025-03-14 13:14:21,627] Trial 4 finished with value: 0.008721466533993591 and parameters: {'num_leaves': 146, 'subsample': 0.312390630426552, 'colsample_bytree': 0.2728131252965228, 'min_data_in_leaf': 79}. Best is trial 1 with value: 0.004740695291357183.
[I 2025-03-14 13:14:21,677] Trial 5 finished with value: 0.004467046617714956 and parameters: {'num_leaves': 74, 'subsample': 0.13362207865112347, 'colsample_bytree': 0.3994696581772982, 'min_data_in_leaf': 35}. Bes

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

[I 2025-03-14 13:14:21,741] Trial 6 finished with value: 0.005036326419022664 and parameters: {'num_leaves': 445, 'subsample': 0.5037103349350125, 'colsample_bytree': 0.6287563255006724, 'min_data_in_leaf': 30}. Best is trial 5 with value: 0.004467046617714956.
[I 2025-03-14 13:14:21,788] Trial 7 finished with value: 0.006478388293421528 and parameters: {'num_leaves': 740, 'subsample': 0.7303228141739868, 'colsample_bytree': 0.9595860469872765, 'min_data_in_leaf': 60}. Best is trial 5 with value: 0.004467046617714956.
[I 2025-03-14 13:14:21,823] Trial 8 finished with value: 0.009603821260693271 and parameters: {'num_leaves': 911, 'subsample': 0.8685344808913239, 'colsample_bytree': 0.25377613332102544, 'min_data_in_leaf': 97}. Best is trial 5 with value: 0.004467046617714956.
[I 2025-03-14 13:14:21,850] Trial 9 finished with value: 0.010546875321777102 and parameters: {'num_leaves': 770, 'subsample': 0.5038184089701058, 'colsample_bytree': 0.24385808778389745, 'min_data_in_leaf': 95}. 

...

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

[I 2025-03-14 13:14:24,459] Trial 46 finished with value: 0.004548937570186677 and parameters: {'num_leaves': 269, 'subsample': 0.7389109023260523, 'colsample_bytree': 0.29111734371574216, 'min_data_in_leaf': 39}. Best is trial 31 with value: 0.004264697021551258.
[I 2025-03-14 13:14:24,518] Trial 47 finished with value: 0.0055585439743630425 and parameters: {'num_leaves': 95, 'subsample': 0.5803663321489805, 'colsample_bytree': 0.5824488294832445, 'min_data_in_leaf': 51}. Best is trial 31 with value: 0.004264697021551258.
[I 2025-03-14 13:14:24,576] Trial 48 finished with value: 0.005291606404421582 and parameters: {'num_leaves': 671, 'subsample': 0.5033318979486248, 'colsample_bytree': 0.14828736097351022, 'min_data_in_leaf': 36}. Best is trial 31 with value: 0.004264697021551258.


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

[I 2025-03-14 13:14:24,658] Trial 49 finished with value: 0.005352928204062415 and parameters: {'num_leaves': 197, 'subsample': 0.2728429196941835, 'colsample_bytree': 0.5035965839303896, 'min_data_in_leaf': 29}. Best is trial 31 with value: 0.004264697021551258.


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

### Random Forest

In [37]:
def objective(trial):
    # Definir los hiperparámetros a optimizar
    n_estimators = trial.suggest_int("n_estimators", 100, 500, step=50)
    max_depth = trial.suggest_int("max_depth", 10, 50, step=5)
    min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
    min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 10)
    bootstrap = trial.suggest_categorical("bootstrap", [True, False])

    # Definir el modelo con los hiperparámetros actuales
    RF_model = RandomForestRegressor(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        bootstrap=bootstrap,
        criterion="squared_error",
        random_state=0
    )

    # Entrenar el modelo
    RF_model.fit(X_train_scaled_df, y_train_scaled_df)

    # Predicciones en el conjunto de validación
    y_pred = RF_model.predict(X_val_scaled_df)
    rmse = mean_squared_error(y_val_scaled_df, y_pred)

    return rmse  # Optuna minimizará este valor

# Crear el estudio de optimización
RF_study = optuna.create_study(direction="minimize")
RF_study.optimize(objective, n_trials=50)  # Ejecutar 50 pruebas

# Imprimir los mejores hiperparámetros encontrados
print("Mejores hiperparámetros:", RF_study.best_params)

[I 2025-03-14 13:14:24,666] A new study created in memory with name: no-name-cdb2205c-6ecb-4bf4-b5cb-bc6c1573a10e
[I 2025-03-14 13:14:25,676] Trial 0 finished with value: 0.0066586504731212595 and parameters: {'n_estimators': 250, 'max_depth': 20, 'min_samples_split': 12, 'min_samples_leaf': 8, 'bootstrap': True}. Best is trial 0 with value: 0.0066586504731212595.
[I 2025-03-14 13:14:27,297] Trial 1 finished with value: 0.008377841402044416 and parameters: {'n_estimators': 300, 'max_depth': 10, 'min_samples_split': 8, 'min_samples_leaf': 4, 'bootstrap': False}. Best is trial 0 with value: 0.0066586504731212595.
[I 2025-03-14 13:14:29,823] Trial 2 finished with value: 0.00895736141929348 and parameters: {'n_estimators': 450, 'max_depth': 50, 'min_samples_split': 16, 'min_samples_leaf': 5, 'bootstrap': False}. Best is trial 0 with value: 0.0066586504731212595.
[I 2025-03-14 13:14:31,537] Trial 3 finished with value: 0.009251471933960599 and parameters: {'n_estimators': 300, 'max_depth': 

Mejores hiperparámetros: {'n_estimators': 100, 'max_depth': 15, 'min_samples_split': 20, 'min_samples_leaf': 10, 'bootstrap': True}


### CTNET

In [38]:
def compile_and_fit(model, xtrain=X_train_windowed, ytrain=y_train_windowed, learning_rate=0.0001):
    model.compile(loss=[tf.keras.losses.MeanSquaredError()],
                  optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
                  metrics=[tf.keras.metrics.RootMeanSquaredError(), tf.keras.metrics.MeanAbsolutePercentageError(), tf.keras.metrics.MeanAbsoluteError()])

    history = model.fit(xtrain, ytrain, epochs=50,
                        batch_size=512, validation_split=0.2, verbose=1)
    return history

def Loss(train_loss, valid_loss):
    plt.plot(train_loss)
    plt.plot(valid_loss)
    plt.rcParams["figure.figsize"] = (15, 3)
    plt.title('Model Losses')
    plt.ylabel('Loss')
    plt.xlabel('Epoch')
    plt.legend(['Train Loss', 'Validation Loss'], loc='upper left')
    plt.savefig('out/loss_plot.png')
    plt.show()

def transformer_encoder(inputs, head_size, num_heads, ff_dim, dropout=0):
    x = layers.LayerNormalization()(inputs)
    x = layers.Conv1D(filters=ff_dim, kernel_size=1, activation="relu", padding="same")(x)
    x = layers.Conv1D(filters=128, kernel_size=2, activation="relu", padding="same")(x)
    x = layers.Conv1D(filters=inputs.shape[-1], kernel_size=1)(x)
    res = x + inputs
    norm_x = layers.LayerNormalization()(res)
    x = layers.MultiHeadAttention(key_dim=head_size, num_heads=num_heads, dropout=dropout)(norm_x, norm_x)
    res = x + inputs
    norm_x = layers.LayerNormalization()(res)
    return norm_x

# Construir modelo con hiperparámetros sugeridos
def build_model(input_shape, head_size, num_heads, ff_dim, num_transformer_blocks, mlp_units, dropout, mlp_dropout):
    inputs = tf.keras.Input(shape=input_shape)
    x = inputs

    for _ in range(num_transformer_blocks):
        x = transformer_encoder(x, head_size, num_heads, ff_dim, dropout)

    x = layers.MultiHeadAttention(key_dim=head_size, num_heads=num_heads, dropout=dropout)(x, x)
    x = layers.LayerNormalization(epsilon=1e-6)(x)
    x = layers.GlobalAveragePooling1D(data_format="channels_first")(x)

    for units in mlp_units:
        x = layers.Dense(units, activation="relu")(x)

    x = layers.Dropout(mlp_dropout)(x)
    outputs = layers.Dense(1)(x)
    return tf.keras.Model(inputs, outputs)

In [39]:
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

In [40]:
# Definir la función objetivo para Optuna
def objective(trial):
    # Hiperparámetros a optimizar
    head_size = trial.suggest_int("head_size", 2, 8)
    num_heads = trial.suggest_int("num_heads", 2, 8)
    ff_dim = trial.suggest_int("ff_dim", 16, 128, step=16)
    num_transformer_blocks = trial.suggest_int("num_transformer_blocks", 1, 5)
    mlp_units = [trial.suggest_int("mlp_units_1", 64, 512, step=64),
                 trial.suggest_int("mlp_units_2", 32, 256, step=32)]
    dropout = trial.suggest_float("dropout", 0.1, 0.5)
    mlp_dropout = trial.suggest_float("mlp_dropout", 0.1, 0.5)
    learning_rate = trial.suggest_loguniform("learning_rate", 1e-5, 1e-2)
    batch_size = trial.suggest_categorical("batch_size", [128, 256, 512])

    # Construir el modelo
    model = build_model(
        input_shape=(X_train_windowed.shape[1], X_train_windowed.shape[2]),
        head_size=head_size,
        num_heads=num_heads,
        ff_dim=ff_dim,
        num_transformer_blocks=num_transformer_blocks,
        mlp_units=mlp_units,
        dropout=dropout,
        mlp_dropout=mlp_dropout
    )

    # Compilar y entrenar el modelo
    model.compile(
        loss=tf.keras.losses.MeanSquaredError(),
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        metrics=[tf.keras.metrics.RootMeanSquaredError()]
    )

    history = model.fit(
        X_train_windowed, y_train_windowed,
        epochs=100,  # Se puede detener antes con Early Stopping
        batch_size=batch_size,  # 🔹 Batch size optimizado
        validation_data=(X_val_windowed, y_val_windowed),  # 🔹 Validación en cada época
        callbacks=[early_stop],  # 🔹 Early Stopping activado
        verbose=0
    )

    # Obtener pérdida de validación
    val_loss = min(history.history['val_loss'])
    return val_loss  # Minimizar MSE

# Crear el estudio de optimización
CTNET_study = optuna.create_study(direction="minimize")
CTNET_study.optimize(objective, n_trials=50)  # Ejecutar 50 pruebas

# Imprimir los mejores hiperparámetros encontrados
print("Mejores hiperparámetros:", CTNET_study.best_params)


[I 2025-03-14 13:15:10,267] A new study created in memory with name: no-name-c3f15ff9-8492-430e-b588-6ca0a5d727a9
[I 2025-03-14 13:15:57,559] Trial 0 finished with value: 0.03725304827094078 and parameters: {'head_size': 5, 'num_heads': 2, 'ff_dim': 32, 'num_transformer_blocks': 2, 'mlp_units_1': 128, 'mlp_units_2': 64, 'dropout': 0.3307771573464182, 'mlp_dropout': 0.21565515523620615, 'learning_rate': 0.007545200558339774, 'batch_size': 128}. Best is trial 0 with value: 0.03725304827094078.
[I 2025-03-14 13:16:40,580] Trial 1 finished with value: 0.038395918905735016 and parameters: {'head_size': 7, 'num_heads': 5, 'ff_dim': 16, 'num_transformer_blocks': 1, 'mlp_units_1': 384, 'mlp_units_2': 160, 'dropout': 0.46903130017863226, 'mlp_dropout': 0.3643478034765427, 'learning_rate': 0.0037879072333974564, 'batch_size': 128}. Best is trial 0 with value: 0.03725304827094078.
[I 2025-03-14 13:17:45,538] Trial 2 finished with value: 0.031527165323495865 and parameters: {'head_size': 6, 'num_h

Mejores hiperparámetros: {'head_size': 2, 'num_heads': 3, 'ff_dim': 128, 'num_transformer_blocks': 5, 'mlp_units_1': 512, 'mlp_units_2': 128, 'dropout': 0.2472431288474223, 'mlp_dropout': 0.12827737593130425, 'learning_rate': 0.00018118580392084425, 'batch_size': 256}


### Forescasting

In [42]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import InputLayer, Conv1D, BatchNormalization, MaxPooling1D, Bidirectional, LSTM, Dropout, Dense
from tensorflow.keras.optimizers import Adam

# Definir la función objetivo para Optuna
def objective(trial):
    # Hiperparámetros a optimizar
    filters = trial.suggest_categorical("filters", [32, 64, 128])
    kernel_size = trial.suggest_int("kernel_size", 2, 5)
    lstm_units_1 = trial.suggest_categorical("lstm_units_1", [64, 128, 256])
    lstm_units_2 = trial.suggest_categorical("lstm_units_2", [32, 64, 128])
    lstm_units_3 = trial.suggest_categorical("lstm_units_3", [16, 32, 64])
    dropout_lstm = trial.suggest_float("dropout_lstm", 0.1, 0.5)
    dropout_dense = trial.suggest_float("dropout_dense", 0.1, 0.5)
    learning_rate = trial.suggest_loguniform("learning_rate", 1e-4, 1e-2)
    batch_size = trial.suggest_categorical("batch_size", [128, 256, 512])  # 🔹 Optimización del batch size

    # Construcción del modelo
    model = Sequential()
    model.add(InputLayer((X_train_windowed.shape[1], X_train_windowed.shape[2])))

    # CNN
    model.add(Conv1D(filters=filters, kernel_size=kernel_size, padding='same', activation='relu'))
    model.add(BatchNormalization())
    model.add(MaxPooling1D(pool_size=2))

    # BiLSTM
    model.add(Bidirectional(LSTM(lstm_units_1, return_sequences=True)))
    model.add(Bidirectional(LSTM(lstm_units_2, return_sequences=True)))
    model.add(Dropout(dropout_lstm))
    model.add(Bidirectional(LSTM(lstm_units_3, return_sequences=False)))

    # Normalización y Dropout
    model.add(BatchNormalization())
    model.add(Dropout(dropout_dense))

    # Capas Densas
    model.add(Dense(16, activation='relu'))
    model.add(Dense(1, activation='relu'))

    # Compilar el modelo
    model.compile(
        loss=tf.keras.losses.MeanSquaredError(),
        optimizer=Adam(learning_rate=learning_rate),
        metrics=[tf.keras.metrics.RootMeanSquaredError()]
    )

    # Entrenar el modelo con validación
    history = model.fit(
        X_train_windowed, y_train_windowed,
        epochs=100,  # Se puede detener antes con Early Stopping
        batch_size=batch_size,  # 🔹 Batch size optimizado
        validation_data=(X_val_windowed, y_val_windowed),  # 🔹 Validación en cada época
        callbacks=[early_stop],  # 🔹 Early Stopping activado
        verbose=0
    )

    # Obtener la mejor pérdida de validación
    val_loss = min(history.history['val_loss'])
    return val_loss  # Optuna minimizará este valor

# Crear el estudio de optimización
Forecasting_study = optuna.create_study(direction="minimize")
Forecasting_study.optimize(objective, n_trials=50, n_jobs=-1)  # Ejecutar 50 pruebas

# Imprimir los mejores hiperparámetros encontrados
print("Mejores hiperparámetros:", Forecasting_study.best_params)

[I 2025-03-14 14:53:17,919] A new study created in memory with name: no-name-f5673ae0-e336-4d25-a5f6-9b0f210e1885
[I 2025-03-14 14:54:32,538] Trial 2 finished with value: 0.3545307517051697 and parameters: {'filters': 64, 'kernel_size': 2, 'lstm_units_1': 256, 'lstm_units_2': 32, 'lstm_units_3': 16, 'dropout_lstm': 0.17568034039292116, 'dropout_dense': 0.46761337700881744, 'learning_rate': 0.0012343560798902444, 'batch_size': 512}. Best is trial 2 with value: 0.3545307517051697.
[I 2025-03-14 14:54:44,837] Trial 12 finished with value: 0.6219170093536377 and parameters: {'filters': 32, 'kernel_size': 2, 'lstm_units_1': 256, 'lstm_units_2': 32, 'lstm_units_3': 16, 'dropout_lstm': 0.10092093730468005, 'dropout_dense': 0.4907074614395538, 'learning_rate': 0.00021361163953979083, 'batch_size': 128}. Best is trial 2 with value: 0.3545307517051697.
[I 2025-03-14 14:54:56,348] Trial 13 finished with value: 0.543649435043335 and parameters: {'filters': 64, 'kernel_size': 5, 'lstm_units_1': 64,

Mejores hiperparámetros: {'filters': 32, 'kernel_size': 4, 'lstm_units_1': 128, 'lstm_units_2': 128, 'lstm_units_3': 64, 'dropout_lstm': 0.31313216228237056, 'dropout_dense': 0.33555840743894466, 'learning_rate': 0.009948373853523741, 'batch_size': 128}


### Photovoltaic

In [43]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv1D, MaxPooling1D, Bidirectional, GRU, MultiHeadAttention
from tensorflow.keras.layers import Flatten, Dropout, Dense
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import mean_squared_error

# Definir la función objetivo para Optuna
def objective(trial):
    # Hiperparámetros a optimizar
    filters_1 = trial.suggest_categorical("filters_1", [32, 64, 128])
    filters_2 = trial.suggest_categorical("filters_2", [64, 128, 256])
    kernel_size = trial.suggest_int("kernel_size", 2, 5)
    gru_units = trial.suggest_categorical("gru_units", [32, 64, 128])
    num_heads = trial.suggest_categorical("num_heads", [2, 4, 8])
    dropout_rate = trial.suggest_float("dropout_rate", 0.2, 0.5)
    dense_units_1 = trial.suggest_categorical("dense_units_1", [32, 64, 128])
    dense_units_2 = trial.suggest_categorical("dense_units_2", [16, 32, 64])
    learning_rate = trial.suggest_loguniform("learning_rate", 1e-4, 1e-2)
    batch_size = trial.suggest_categorical("batch_size", [128, 256, 512])

    # Definir el modelo
    inputs = Input(shape=(X_train_windowed.shape[1], X_train_windowed.shape[2]))

    # Primera capa CNN
    x = Conv1D(filters=filters_1, kernel_size=kernel_size, padding='same', activation='relu')(inputs)
    x = MaxPooling1D(pool_size=2)(x)

    # Segunda capa CNN
    x = Conv1D(filters=filters_2, kernel_size=kernel_size, padding='same', activation='relu')(x)
    x = MaxPooling1D(pool_size=2)(x)

    # Capa BiGRU
    x = Bidirectional(GRU(gru_units, return_sequences=True))(x)

    # Atención MultiHead
    attention = MultiHeadAttention(num_heads=num_heads, key_dim=128)(x, x)

    # Aplanar y Dropout
    x = Flatten()(attention)
    x = Dropout(dropout_rate)(x)
    x = Dense(dense_units_1, activation="relu", kernel_regularizer=l2(0.01))(x)
    x = Dense(dense_units_2, activation="relu")(x)

    # Capa de salida
    outputs = Dense(1, activation="linear")(x)

    # Construcción del modelo
    model = Model(inputs=inputs, outputs=outputs)

    # Compilar el modelo
    model.compile(
        loss=tf.keras.losses.MeanSquaredError(),
        optimizer=Adam(learning_rate=learning_rate),
        metrics=[tf.keras.metrics.RootMeanSquaredError()]
    )

    # Early Stopping
    early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1)

    # Entrenar el modelo
    history = model.fit(
        X_train_windowed, y_train_windowed,
        epochs=100,  # Se detendrá con Early Stopping
        batch_size=batch_size,
        validation_data=(X_val_windowed, y_val_windowed),
        callbacks=[early_stop],
        verbose=0
    )

    # Obtener la mejor pérdida de validación
    val_loss = min(history.history['val_loss'])
    return val_loss  # Optuna minimizará este valor

# Crear el estudio de optimización
Photovoltaic_study = optuna.create_study(direction="minimize")
Photovoltaic_study.optimize(objective, n_trials=50, n_jobs=-1)  # Ejecutar 50 pruebas

# Imprimir los mejores hiperparámetros encontrados
print("Mejores hiperparámetros:", Photovoltaic_study.best_params)

[I 2025-03-14 15:04:08,392] A new study created in memory with name: no-name-f7967b61-92f3-49c0-9b8f-1bc27c47d0b9


Epoch 62: early stopping
Restoring model weights from the end of the best epoch: 52.


[I 2025-03-14 15:05:33,653] Trial 5 finished with value: 0.036089938133955 and parameters: {'filters_1': 64, 'filters_2': 256, 'kernel_size': 4, 'gru_units': 128, 'num_heads': 4, 'dropout_rate': 0.3976615086038989, 'dense_units_1': 64, 'dense_units_2': 16, 'learning_rate': 0.0009830132945336009, 'batch_size': 512}. Best is trial 5 with value: 0.036089938133955.


Restoring model weights from the end of the best epoch: 100.


[I 2025-03-14 15:05:51,098] Trial 3 finished with value: 0.1698700189590454 and parameters: {'filters_1': 128, 'filters_2': 64, 'kernel_size': 4, 'gru_units': 64, 'num_heads': 4, 'dropout_rate': 0.3051737098213132, 'dense_units_1': 128, 'dense_units_2': 32, 'learning_rate': 0.0002104973425293171, 'batch_size': 512}. Best is trial 5 with value: 0.036089938133955.


...

Restoring model weights from the end of the best epoch: 100.


[I 2025-03-14 15:11:27,333] Trial 47 finished with value: 0.15621808171272278 and parameters: {'filters_1': 128, 'filters_2': 128, 'kernel_size': 2, 'gru_units': 128, 'num_heads': 4, 'dropout_rate': 0.23032426145241647, 'dense_units_1': 64, 'dense_units_2': 64, 'learning_rate': 0.004132768179733768, 'batch_size': 128}. Best is trial 28 with value: 0.013302194885909557.


Restoring model weights from the end of the best epoch: 100.


[I 2025-03-14 15:11:28,157] Trial 48 finished with value: 0.1174120157957077 and parameters: {'filters_1': 128, 'filters_2': 128, 'kernel_size': 2, 'gru_units': 128, 'num_heads': 4, 'dropout_rate': 0.22382659597022647, 'dense_units_1': 128, 'dense_units_2': 16, 'learning_rate': 0.0047141664062042695, 'batch_size': 128}. Best is trial 28 with value: 0.013302194885909557.


Mejores hiperparámetros: {'filters_1': 128, 'filters_2': 128, 'kernel_size': 2, 'gru_units': 32, 'num_heads': 4, 'dropout_rate': 0.2411091331641294, 'dense_units_1': 64, 'dense_units_2': 64, 'learning_rate': 0.00224494896517205, 'batch_size': 128}
